# 📚 Task 1: Web Scraping Book Data

## Objective

The objective of this task is to collect book information from the **BooksToScrape** website using Python web scraping techniques.

The extracted data includes:

- Book Title
- Price
- Star Rating
- Availability Status
- Product URL

## Tools & Libraries

- Python
- Requests
- BeautifulSoup
- Pandas

## Workflow

1. Send HTTP requests to the website.
2. Parse HTML using BeautifulSoup.
3. Extract required book information.
4. Store the data in a structured DataFrame.
5. Save the dataset as a CSV file for further analysis.

## Output

- `books_dataset.csv`

This dataset will be used in the next tasks for data cleaning, visualization, and sentiment analysis.

## Setup cell

In [ ]:
# Install dependencies (run once)
!pip install requests beautifulsoup4 pandas numpy scipy matplotlib seaborn vaderSentiment textblob -q

import os
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("visuals", exist_ok=True)


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import requests
import pandas as pd
from bs4 import BeautifulSoup

os.makedirs("data/raw", exist_ok=True)


# Task 1 — Web Scraping

In this task, book information is extracted from the public website
BooksToScrape using BeautifulSoup.

The following attributes are collected:

- Title
- Price
- Rating
- Availability
- Product URL

The scraped dataset is stored for further analysis.

In [3]:
import time
import logging
import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/"
START_URL = "https://books.toscrape.com/catalogue/page-1.html"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) BookAnalyticsBot/1.0"}
RATING_WORD_TO_INT = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("scraper")


def fetch_page(url, session):
    """Fetch a URL and return a parsed BeautifulSoup object, or None on failure."""
    try:
        response = session.get(url, headers=HEADERS, timeout=10)
        response.raise_for_status()
    except requests.RequestException as exc:
        logger.warning(f"Failed to fetch {url} ({exc})")
        return None
    return BeautifulSoup(response.text, "html.parser")


def parse_book_card(card):
    """Extract structured fields from a single '.product_pod' <article> tag."""
    title = card.h3.a["title"].strip()

    price_text = card.find("p", class_="price_color").get_text(strip=True)
    price_gbp = float(price_text.replace("£", "").replace("Â", "").strip())

    availability = card.find("p", class_="instock availability").get_text(strip=True)

    rating_classes = card.find("p", class_="star-rating")["class"]
    rating_word = rating_classes[1] if len(rating_classes) > 1 else "Zero"
    rating = RATING_WORD_TO_INT.get(rating_word, 0)

    detail_href = card.h3.a["href"]
    detail_url = BASE_URL + detail_href.replace("../../../", "").replace("../../", "")

    return {
        "Title": title,
        "Price": price_gbp,
        "Availability": availability,
        "Rating": rating,
        "DetailURL": detail_url,
    }


def crawl_catalogue(start_url=START_URL, max_pages=None):
    """Walk every paginated catalogue page and return a list of book dicts.
    Set max_pages (e.g. 5) to limit the crawl for a quick test run."""
    books = []
    current_url = start_url
    session = requests.Session()
    page_count = 0

    while current_url:
        page_count += 1
        logger.info(f"Scraping page {page_count}: {current_url}")

        soup = fetch_page(current_url, session)
        if soup is None:
            break

        for card in soup.find_all("article", class_="product_pod"):
            books.append(parse_book_card(card))

        next_link = soup.find("li", class_="next")
        if next_link and next_link.a and (max_pages is None or page_count < max_pages):
            current_url = BASE_URL + next_link.a["href"]
            time.sleep(1)  # polite delay
        else:
            current_url = None

    logger.info(f"Finished crawling {page_count} page(s), {len(books)} books.")
    return books


# Run the scraper
books = crawl_catalogue(max_pages=50)

df_books = pd.DataFrame(books)
df_books.to_csv("data/raw/books_dataset.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df_books)} books to data/raw/books_dataset.csv")
df_books.head()

2026-07-17 22:17:53,186 | INFO | Scraping page 1: https://books.toscrape.com/catalogue/page-1.html
2026-07-17 22:17:55,457 | INFO | Scraping page 2: https://books.toscrape.com/catalogue/page-2.html
2026-07-17 22:17:57,219 | INFO | Scraping page 3: https://books.toscrape.com/catalogue/page-3.html
2026-07-17 22:17:58,910 | INFO | Scraping page 4: https://books.toscrape.com/catalogue/page-4.html
2026-07-17 22:18:00,686 | INFO | Scraping page 5: https://books.toscrape.com/catalogue/page-5.html
2026-07-17 22:18:02,539 | INFO | Scraping page 6: https://books.toscrape.com/catalogue/page-6.html
2026-07-17 22:18:04,480 | INFO | Scraping page 7: https://books.toscrape.com/catalogue/page-7.html
2026-07-17 22:18:06,198 | INFO | Scraping page 8: https://books.toscrape.com/catalogue/page-8.html
2026-07-17 22:18:08,004 | INFO | Scraping page 9: https://books.toscrape.com/catalogue/page-9.html
2026-07-17 22:18:09,773 | INFO | Scraping page 10: https://books.toscrape.com/catalogue/page-10.html
2026-07-

Saved 1000 books to data/raw/books_dataset.csv


,Title,Price,Availability,Rating,DetailURL
0,A Light in the Attic,51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...


In [5]:
from pathlib import Path

# Create folder if it doesn't exist
Path("data/raw").mkdir(parents=True, exist_ok=True)

# Create DataFrame
df_books = pd.DataFrame(books)

# Save CSV
df_books.to_csv(
    "data/raw/books_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved {len(df_books)} books to data/raw/books_dataset.csv")
print(df_books.shape)

# Preview dataset
df_books.head()

Saved 1000 books to data/raw/books_dataset.csv
(1000, 5)


,Title,Price,Availability,Rating,DetailURL
0,A Light in the Attic,51.77,In stock,3,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,In stock,1,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,In stock,1,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,In stock,4,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,In stock,5,https://books.toscrape.com/catalogue/sapiens-a...


In [8]:
df_books.to_csv("data/raw/books_dataset.csv", index=False, encoding="utf-8-sig")

## Conclusion

The web scraping process successfully extracted book information from the BooksToScrape website using Python and BeautifulSoup. The collected dataset contains important attributes such as book title, price, rating, availability, and product URL. The extracted data was stored in CSV format, providing a structured dataset for further preprocessing and analysis.

### Key Outcome

- Successfully scraped book data from multiple web pages.
- Created a structured dataset for analysis.
- Saved the dataset in CSV format for future tasks.
## Future Improvements

- Scrape additional book details such as category and product description.
- Implement asynchronous scraping to improve performance.
- Add retry mechanisms for failed requests.